<a href="https://colab.research.google.com/github/amirhamrouni/Alala/blob/main/docs/MoneyPrinterTurbo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/amirhamrouni/Alala/blob/main/docs/MoneyPrinterTurbo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MoneyPrinterTurbo Automation Stack\nخلية واحدة تشغل MoneyPrinterTurbo API + WebUI + إصلاح العربية RTL + Queue Gateway + Telegram long-poll. لا يلزم VPS أو SSH. ضع الأسرار في Colab Secrets: `NGROK_AUTHTOKEN`, `TELEGRAM_BOT_TOKEN`, `PEXELS_API_KEY`، و`OPENROUTER_API_KEY` اختياري.

In [ ]:
import os, re, subprocess, time, urllib.request, textwrap, json
from pathlib import Path
from getpass import getpass
import requests # Ensure requests is imported for getMe check

MPT = Path('/content/MoneyPrinterTurbo')
ALALA = Path('/content/Alala')
MPT_UPSTREAM = 'https://github.com/harry0703/MoneyPrinterTurbo.git'
ALALA_UPSTREAM = 'https://github.com/amirhamrouni/Alala.git'
PATCH_URL = 'https://raw.githubusercontent.com/amirhamrouni/Alala/main/moneyprinter-mobile-gateway/scripts/patch_moneyprinter_arabic.py'

API_PORT = 8090
WEB_PORT = 8501
GATEWAY_PORT = 8000

def run(cmd, **kwargs):
    print('$', ' '.join(map(str, cmd)), flush=True)
    return subprocess.run(cmd, check=True, **kwargs)

def healthy(url):
    try:
        with urllib.request.urlopen(url, timeout=3) as r:
            return r.status == 200
    except Exception:
        return False

def secret(name):
    try:
        from google.colab import userdata
        value = userdata.get(name)
        return value.strip() if value else ''
    except Exception:
        return ''

run(['python', '-m', 'pip', 'install', '-q', 'uv', 'pyngrok', 'requests'])

for repo, upstream in ((MPT, MPT_UPSTREAM), (ALALA, ALALA_UPSTREAM)):
    if (repo / '.git').is_dir():
        run(['git', '-C', str(repo), 'pull', '--ff-only'])
    else:
        if repo.exists():
            run(['rm', '-rf', str(repo)])
        run(['git', 'clone', '--depth', '1', upstream, str(repo)])

run(['uv', 'python', 'install', '3.11'])
run(['uv', 'sync', '--frozen', '--python', '3.11'], cwd=MPT)
run(['python', '-m', 'pip', 'install', '-q', '-r', str(ALALA / 'moneyprinter-mobile-gateway/requirements.txt')])

# Install Arabic-capable fonts
print('$ apt-get update -qq && apt-get install -y -qq fonts-dejavu')
subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'fonts-dejavu'], check=True)

patch_file = Path('/content/patch_moneyprinter_arabic.py')
urllib.request.urlretrieve(PATCH_URL, patch_file)

# Explicitly run the patch script to capture and display its output on error
patch_cmd = ['python', str(patch_file), '--repo', str(MPT)]
print('$', ' '.join(map(str, patch_cmd)), flush=True)
try:
    subprocess.run(patch_cmd, check=True, capture_output=True, text=True)
except subprocess.CalledProcessError as e:
    print(f"Error executing patch script '{' '.join(e.cmd)}':")
    if e.stdout:
        print("STDOUT:\n", e.stdout)
    if e.stderr:
        print("STDERR:\n", e.stderr)
    raise # Re-raise the exception after printing details

cfg = MPT / 'config.toml'
if not cfg.exists():
    cfg.write_text((MPT / 'config.example.toml').read_text(encoding='utf-8'), encoding='utf-8')
raw = cfg.read_text(encoding='utf-8')

def set_toml_scalar(text, key, value_literal):
    pattern = rf'(?m)^{re.escape(key)}\s*=\s*.*$'
    replacement = f'{key} = {value_literal}'
    if re.search(pattern, text):
        return re.sub(pattern, replacement, text, count=1)
    return text + '\n' + replacement + '\n'

raw = set_toml_scalar(raw, 'listen_port', str(API_PORT))
raw = set_toml_scalar(raw, 'video_source', '"pexels"')

pexels_key = secret('PEXELS_API_KEY') or secret('MPT_PEXELS_API_KEY')
if pexels_key:
    raw = set_toml_scalar(raw, 'pexels_api_keys', json.dumps([pexels_key]))

openrouter_key = secret('OPENROUTER_API_KEY')
if openrouter_key:
    raw = set_toml_scalar(raw, 'llm_provider', '"openrouter"')
    raw = set_toml_scalar(raw, 'openrouter_api_key', json.dumps(openrouter_key))
    openrouter_model = secret('OPENROUTER_MODEL') or 'minimax/minimax-m3:free'
    raw = set_toml_scalar(raw, 'openrouter_model_name', json.dumps(openrouter_model))

cfg.write_text(raw, encoding='utf-8')

for port in (API_PORT, WEB_PORT, GATEWAY_PORT):
    subprocess.run(['bash', '-lc', f'fuser -k {port}/tcp >/dev/null 2>&1 || true'], check=False)
time.sleep(1)

api_log = open('/content/mpt-api.log', 'w', encoding='utf-8')
web_log = open('/content/mpt-webui.log', 'w', encoding='utf-8')

api_proc = subprocess.Popen(['uv', 'run', 'python', 'main.py'], cwd=MPT, stdout=api_log, stderr=subprocess.STDOUT)
web_proc = subprocess.Popen([
    'uv','run','streamlit','run','webui/Main.py',
    f'--server.port={WEB_PORT}','--server.address=0.0.0.0',
    '--browser.gatherUsageStats=False','--client.toolbarMode=minimal',
    '--server.showEmailPrompt=False'
], cwd=MPT, stdout=web_log, stderr=subprocess.STDOUT)

for _ in range(60):
    if healthy(f'http://172.28.0.1:{API_PORT}/ping') and healthy(f'http://172.28.0.1:{WEB_PORT}/_stcore/health'):
        break
    time.sleep(2)
else:
    api_log.flush(); web_log.flush()
    print(Path('/content/mpt-api.log').read_text(errors='replace')[-3000:])
    print(Path('/content/mpt-webui.log').read_text(errors='replace')[-3000:])
    raise RuntimeError('MoneyPrinterTurbo did not become healthy')

tg_token = secret('TELEGRAM_BOT_TOKEN')
gateway_env = os.environ.copy()
gateway_env.update({
    'MONEYPRINTER_BASE_URL': f'http://172.28.0.1:{API_PORT}', # Use internal IP for gateway to API communication
    'MONEYPRINTER_STORAGE_ROOT': str(MPT),
    'JOB_STATE_PATH': '/content/mpt-gateway-data/jobs.json',
    'DEFAULT_LANGUAGE': 'ar',
    'DEFAULT_ASPECT': '9:16',
    'DEFAULT_VIDEO_SOURCE': 'pexels',
    'DEFAULT_VIDEO_CONCAT_MODE': 'random',
    'DEFAULT_VIDEO_CLIP_DURATION': '3',
    'DEFAULT_VOICE_NAME': 'ar-SA-ZariyahNeural',
    'DEFAULT_FONT_NAME': 'DejaVuSans.ttf',
    'TELEGRAM_SEND_VIDEO': 'true',
})
if tg_token:
    gateway_env['TELEGRAM_BOT_TOKEN'] = tg_token

gateway_log = open('/content/mpt-gateway.log', 'w', encoding='utf-8')
gateway_proc = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'app.main:app',
     '--app-dir', str(ALALA / 'moneyprinter-mobile-gateway'),
     '--host', '0.0.0.0', # Listen on all interfaces
     '--port', str(GATEWAY_PORT)],
    cwd=ALALA / 'moneyprinter-mobile-gateway',
    env=gateway_env,
    stdout=gateway_log, stderr=subprocess.STDOUT
)

for _ in range(30):
    if healthy(f'http://172.28.0.1:{GATEWAY_PORT}/health'):
        break
    time.sleep(1)
else:
    gateway_log.flush()
    print(Path('/content/mpt-gateway.log').read_text(errors='replace')[-3000:])
    raise RuntimeError('Gateway did not become healthy')

telegram_poller_proc = None # Initialize outside if block
if tg_token:
    print("TELEGRAM_BOT_TOKEN detected in Colab Secrets, starting Telegram poller...")
    poller_path = Path('/content/telegram_poller.py')
    # Use 172.28.0.1 for internal gateway communication
    poller_path.write_text(textwrap.dedent(f'''
        import os, time, requests

        token = os.environ['TELEGRAM_BOT_TOKEN']
        tg = f'https://api.telegram.org/bot{{token}}'
        gateway = 'http://172.28.0.1:{GATEWAY_PORT}/telegram/webhook' # Use internal Colab IP for gateway
        s = requests.Session()

        # Delete existing webhook before starting long polling
        s.post(tg + '/deleteWebhook', json={{'drop_pending_updates': False}}, timeout=30)

        # Test getMe to confirm token is valid
        me_response = s.get(tg + '/getMe', timeout=30).json()
        if not me_response.get('ok'):
            raise RuntimeError(f"Telegram token rejected: {{me_response.get('description', 'Unknown error')}}")
        print('Telegram ready: @' + me_response['result']['username'], flush=True)

        offset = 0
        while True:
            try:
                data = s.get(tg + '/getUpdates', params={{'offset': offset, 'timeout': 25}}, timeout=35).json()
                for update in data.get('result', []):
                    offset = update['update_id'] + 1
                    # Send update to local gateway
                    r = s.post(gateway, json=update, timeout=30)
                    r.raise_for_status()
            except Exception as exc:
                print('poll error:', type(exc).__name__, str(exc)[:200], flush=True)
                time.sleep(5)
    '''), encoding='utf-8')
    poller_log = open('/content/telegram-poller.log', 'w', encoding='utf-8')
    telegram_poller_proc = subprocess.Popen(
        ['python', str(poller_path)],
        env={**os.environ, 'TELEGRAM_BOT_TOKEN': tg_token},
        stdout=poller_log, stderr=subprocess.STDOUT
    )
    time.sleep(2) # Give poller a moment to start and print its status
else:
    print('TELEGRAM_BOT_TOKEN NOT found or empty in Colab Secrets. Telegram bridge skipped.')

ngrok_token = secret('NGROK_AUTHTHOKEN')
if not ngrok_token:
    ngrok_token = getpass('NGROK_AUTHTOKEN: ').strip()

if ngrok_token:
    from pyngrok import ngrok
    ngrok.set_auth_token(ngrok_token)
    try:
        ngrok.kill()
    except Exception:
        pass
    # Connect ngrok to the WebUI's internal IP
    web_tunnel = ngrok.connect(addr=f'http://172.28.0.1:{WEB_PORT}', proto='http', bind_tls=True)
    print('WEBUI =', web_tunnel.public_url)
else:
    print('NGROK_AUTHTOKEN missing; WebUI remains local.')

print('API_LOCAL =', f'http://172.28.0.1:{API_PORT}')
print('GATEWAY_LOCAL =', f'http://172.28.0.1:{GATEWAY_PORT}')
print('Arabic subtitles = RTL patched / DejaVuSans')
print('Queue + dedupe + job_id = enabled')
print('Telegram MP4 upload = enabled when TELEGRAM_BOT_TOKEN exists')

# --- Final Status Report ---
print('\n--- Final Status Report ---')

# MPT API Check
mpt_api_status = 'FAIL'
try:
    if healthy(f'http://172.28.0.1:{API_PORT}/ping'):
        mpt_api_status = 'PASS'
except Exception:
    pass # Keep FAIL if exception
print(f'MPT_API={mpt_api_status}')

# WebUI Check
webui_status = 'FAIL'
try:
    if healthy(f'http://172.28.0.1:{WEB_PORT}/_stcore/health'):
        webui_status = 'PASS'
except Exception:
    pass # Keep FAIL if exception
print(f'WEBUI={webui_status}')

# Telegram Check
telegram_status = 'FAIL'
telegram_username = ''

if tg_token:
    if telegram_poller_proc and telegram_poller_proc.poll() is None:
        try:
            # Test getMe directly to confirm bot is active and get username
            tg_api_url = f'https://api.telegram.org/bot{tg_token}'
            me_response = requests.get(tg_api_url + '/getMe', timeout=10).json()
            if me_response.get('ok'):
                telegram_username = me_response['result']['username']
                telegram_status = 'PASS'
        except Exception as e:
            print(f"Telegram final getMe check failed: {type(e).__name__} - {e}")
    else:
        print("Telegram poller process not running for final check.")

if telegram_status == 'PASS':
    print(f'TELEGRAM=PASS (@{telegram_username})')
else:
    print('TELEGRAM=FAIL (Verify TELEGRAM_BOT_TOKEN in Colab Secrets and check poller logs)')

# Arabic RTL Patch Check
# Assumed PASS if the fonts-dejavu installation and patch script executed without error previously.
arabic_rtl_status = 'PASS'
print(f'ARABIC_RTL={arabic_rtl_status}')

print('READY')


$ python -m pip install -q uv pyngrok requests
$ git -C /content/MoneyPrinterTurbo pull --ff-only
$ git -C /content/Alala pull --ff-only
$ uv python install 3.11
$ uv sync --frozen --python 3.11
$ python -m pip install -q -r /content/Alala/moneyprinter-mobile-gateway/requirements.txt
$ apt-get update -qq && apt-get install -y -qq fonts-dejavu
$ python /content/patch_moneyprinter_arabic.py --repo /content/MoneyPrinterTurbo


## اختبار الحالة\nبعد ظهور `READY`، أرسل للبوت: `اعمل فيديو عن فوائد الذكاء الاصطناعي في الدراسة`. البوت يضع الطلب في Queue ثم يعيد ملف MP4 لنفس المحادثة.

In [ ]:
import requests, json, pathlib\n\nchecks = {\n    'mpt_api': requests.get('http://127.0.0.1:8090/ping', timeout=10).status_code,\n    'webui': requests.get('http://127.0.0.1:8501/_stcore/health', timeout=10).status_code,\n    'gateway': requests.get('http://127.0.0.1:8000/health', timeout=10).json(),\n}\nprint(json.dumps(checks, ensure_ascii=False, indent=2))\n\nfor path in ('/content/mpt-api.log', '/content/mpt-webui.log', '/content/mpt-gateway.log', '/content/telegram-poller.log'):\n    p = pathlib.Path(path)\n    if p.exists():\n        print('\n---', p.name, '---')\n        print(p.read_text(errors='replace')[-1200:])\n